# SWE015 — Real-Time Chat Moderation
Four-model training and comparison notebook. Dataset: Jigsaw Toxic Comment Classification. This notebook documents the experiments; the GitHub `app.py` is the standalone application.


In [ ]:
print("SWE015 Chat Moderation Project")
print("Google Colab is working!")

SWE015 Chat Moderation Project
Google Colab is working!


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
GPU available: True
GPU name: Tesla T4


In [2]:

!pip install datasets transformers accelerate evaluate scikit-learn gensim gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]


In [3]:

import torch
import datasets
import transformers
import gradio
import sklearn

print("All libraries are working!")
print("GPU available:", torch.cuda.is_available())


All libraries are working!
GPU available: True


## 1. Dataset loading and preprocessing


In [4]:

from datasets import load_dataset

# Load the Jigsaw Toxic Comment Dataset
dataset = load_dataset(
    "tcapelle/jigsaw-toxic-comment-classification-challenge",
    split="train"
)

print("Dataset loaded successfully!")
print("Total comments:", len(dataset))
print("Columns:", dataset.column_names)


README.md:   0%|          | 0.00/765 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 44.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 39.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/balanced_test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 2.73MB            

data/balanced_test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/153164 [00:00<?, ? examples/s]

Generating balanced_test split:   0%|          | 0/12486 [00:00<?, ? examples/s]

Dataset loaded successfully!
Total comments: 159571
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [5]:

import pandas as pd

# Convert the dataset into a DataFrame
df = dataset.to_pandas()

# Categories of harmful comments
toxic_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Create a binary label
# 0 = Clean, 1 = Offensive
df["label"] = df[toxic_columns].max(axis=1).astype(int)

# Display dataset information
print("Total comments:", len(df))
print("\nMessage classification:")
print(df["label"].value_counts())

print("\nClean messages:", (df["label"] == 0).sum())
print("Offensive messages:", (df["label"] == 1).sum())

print("\nDataset preparation completed!")


Total comments: 159571

Message classification:
label
0    143346
1     16225
Name: count, dtype: int64

Clean messages: 143346
Offensive messages: 16225

Dataset preparation completed!


In [6]:

from sklearn.model_selection import train_test_split

df = df.dropna(subset=["comment_text"]).copy()
df = df[df["comment_text"].str.strip() != ""]
df = df.drop_duplicates(subset=["comment_text"])
df = df.reset_index(drop=True)

data = df[["comment_text", "label"]]


train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["label"]
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Data split completed successfully!")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())


Data split completed successfully!
Training: 127656
Validation: 15957
Testing: 15958

Training labels:
label
0    114676
1     12980
Name: count, dtype: int64

Testing labels:
label
0    14335
1     1623
Name: count, dtype: int64


In [7]:

import re

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)

    text = re.sub(r"@\w+", " USER ", text)

    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)

    text = re.sub(r"[^a-z0-9\s!?']", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["clean_text"] = train_df["comment_text"].apply(clean_text)
val_df["clean_text"] = val_df["comment_text"].apply(clean_text)
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)

example = "Helloooo!!! Please visit https://example.com"

print("Before:", example)
print("After:", clean_text(example))

print("\nText preprocessing completed!")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))


Before: Helloooo!!! Please visit https://example.com
After: helloo!!! please visit

Text preprocessing completed!
Training: 127656
Validation: 15957
Testing: 15958


## 2. Classical NLP baseline (TF-IDF + Logistic Regression)


In [8]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
import time

tfidf = TfidfVectorizer(
    max_features=30000,
    min_df=3,
    ngram_range=(1, 2)
)

X_train = tfidf.fit_transform(train_df["clean_text"])
X_val = tfidf.transform(val_df["clean_text"])

baseline_model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    random_state=42
)

start_time = time.perf_counter()

baseline_model.fit(X_train, train_df["label"])

training_time = time.perf_counter() - start_time

val_predictions = baseline_model.predict(X_val)

macro_f1 = f1_score(
    val_df["label"],
    val_predictions,
    average="macro"
)

print("Classical NLP Model Training Completed!")
print("Training time:", round(training_time, 2), "seconds")
print("Validation Macro-F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(classification_report(
    val_df["label"],
    val_predictions,
    target_names=["Clean", "Offensive"],
    zero_division=0
))


Classical NLP Model Training Completed!
Training time: 3.96 seconds
Validation Macro-F1: 0.8494

Classification Report:
              precision    recall  f1-score   support

       Clean       0.98      0.95      0.96     14335
   Offensive       0.64      0.85      0.73      1622

    accuracy                           0.94     15957
   macro avg       0.81      0.90      0.85     15957
weighted avg       0.95      0.94      0.94     15957



In [9]:

import re

leet_map = str.maketrans({
    "0": "o",
    "1": "i",
    "3": "e",
    "4": "a",
    "5": "s",
    "7": "t"
})

def normalize_chat(text):
    text = str(text).lower()

    text = re.sub(
        r"\b[a-z0-9]*[a-z][a-z0-9]*\b",
        lambda match: match.group().translate(leet_map),
        text
    )

    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)
    return text

def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, 1):
        current = [i]

        for j, char_b in enumerate(b, 1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j-1] + (char_a != char_b)
            ))

        previous = current

    return previous[-1]

offensive_words = ["idiot", "stupid", "moron"]

def check_offensive_words(message):
    text = normalize_chat(message)
    words = re.findall(r"\b[a-z]+\b", text)
    matches = []

    for word in words:
        for target in offensive_words:
            max_distance = 1 if len(target) >= 5 else 0

            if edit_distance(word, target) <= max_distance:
                matches.append(target)

    return sorted(set(matches))

messages = [
    "You are an idi0t",
    "That was stupiid",
    "Thank you for your help"
]

for message in messages:
    print("Message:", message)
    print("Detected words:", check_offensive_words(message))
    print("--------------------")


Message: You are an idi0t
Detected words: ['idiot']
--------------------
Message: That was stupiid
Detected words: ['stupid']
--------------------
Message: Thank you for your help
Detected words: []
--------------------


In [10]:
def moderate_message(message):
    normalized = normalize_chat(message)
    cleaned = clean_text(normalized)

    detected_words = check_offensive_words(message)

    X = tfidf.transform([cleaned])
    probabilities = baseline_model.predict_proba(X)[0]

    prediction = int(baseline_model.classes_[probabilities.argmax()])
    confidence = probabilities.max() * 100

    result = "Offensive" if prediction == 1 else "Clean"

    displayed_message = (
        "".join("*" if char.isalnum() else char for char in message)
        if prediction == 1 else message
    )

    print("Original message:", message)
    print("Classification:", result)
    print("Confidence:", round(confidence, 2), "%")
    print("Detected words:", detected_words)
    print("Displayed message:", displayed_message)
    print("----------------------------------")


moderate_message("Hello, how are you?")
moderate_message("You are an idi0t")
moderate_message("Thank you for helping me")
moderate_message("That was stupiid")


Original message: Hello, how are you?
Classification: Clean
Confidence: 51.86 %
Detected words: []
Displayed message: Hello, how are you?
----------------------------------
Original message: You are an idi0t
Classification: Offensive
Confidence: 100.0 %
Detected words: ['idiot']
Displayed message: *** *** ** *****
----------------------------------
Original message: Thank you for helping me
Classification: Clean
Confidence: 97.0 %
Detected words: []
Displayed message: Thank you for helping me
----------------------------------
Original message: That was stupiid
Classification: Clean
Confidence: 77.67 %
Detected words: ['stupid']
Displayed message: That was stupiid
----------------------------------


In [11]:


def moderate_message(message):
    normalized = normalize_chat(message)
    cleaned = clean_text(normalized)

    detected_words = check_offensive_words(message)

    X = tfidf.transform([cleaned])
    probabilities = baseline_model.predict_proba(X)[0]

    prediction_index = probabilities.argmax()
    model_prediction = int(
        baseline_model.classes_[prediction_index]
    )
    confidence = probabilities[prediction_index] * 100

    final_prediction = model_prediction

    if detected_words:
        final_prediction = 1

    result = "Offensive" if final_prediction == 1 else "Clean"

    if final_prediction == 1:
        displayed_message = "".join(
            "*" if char.isalnum() else char
            for char in message
        )
    else:
        displayed_message = message

    print("Original message:", message)
    print("Model prediction:",
          "Offensive" if model_prediction == 1 else "Clean")
    print("Model confidence:", round(confidence, 2), "%")
    print("Detected words:", detected_words)
    print("Final classification:", result)
    print("Displayed message:", displayed_message)
    print("----------------------------------")


moderate_message("Hello, how are you?")
moderate_message("You are an idi0t")
moderate_message("Thank you for helping me")
moderate_message("That was stupiid")


Original message: Hello, how are you?
Model prediction: Clean
Model confidence: 51.86 %
Detected words: []
Final classification: Clean
Displayed message: Hello, how are you?
----------------------------------
Original message: You are an idi0t
Model prediction: Offensive
Model confidence: 100.0 %
Detected words: ['idiot']
Final classification: Offensive
Displayed message: *** *** ** *****
----------------------------------
Original message: Thank you for helping me
Model prediction: Clean
Model confidence: 97.0 %
Detected words: []
Final classification: Clean
Displayed message: Thank you for helping me
----------------------------------
Original message: That was stupiid
Model prediction: Clean
Model confidence: 77.67 %
Detected words: ['stupid']
Final classification: Offensive
Displayed message: **** *** *******
----------------------------------


## 3. Self-trained neural network (Word2Vec + feedforward network)


In [12]:

from gensim.models import Word2Vec
import re
import time

train_tokens = [
    re.findall(r"\b[a-z]+\b", text)
    for text in train_df["clean_text"].fillna("")
]

train_tokens = [tokens for tokens in train_tokens if tokens]

print("Preparing Word2Vec training...")
print("Training messages:", len(train_tokens))

start_time = time.perf_counter()

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=2,
    sg=0,
    epochs=5,
    seed=42
)

training_time_w2v = time.perf_counter() - start_time

w2v_model.save("chat_word2vec.model")

print("\nWord2Vec Training Completed!")
print("Vocabulary size:", len(w2v_model.wv))
print("Vector size:", w2v_model.vector_size)
print("Training time:", round(training_time_w2v, 2), "seconds")
print("Model saved successfully!")


Preparing Word2Vec training...
Training messages: 127633

Word2Vec Training Completed!
Vocabulary size: 72022
Vector size: 100
Training time: 65.63 seconds
Model saved successfully!


In [13]:

import numpy as np
import re

def text_to_vector(text):
    words = re.findall(r"\b[a-z]+\b", str(text))

    vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if not vectors:
        return np.zeros(100, dtype=np.float32)

    return np.mean(vectors, axis=0).astype(np.float32)


print("Preparing training vectors...")

X_train_nn = np.array([
    text_to_vector(text)
    for text in train_df["clean_text"]
], dtype=np.float32)

print("Preparing validation vectors...")

X_val_nn = np.array([
    text_to_vector(text)
    for text in val_df["clean_text"]
], dtype=np.float32)

y_train_nn = train_df["label"].to_numpy(dtype=np.int64)
y_val_nn = val_df["label"].to_numpy(dtype=np.int64)

print("\nWord2Vec Features Prepared!")
print("Training shape:", X_train_nn.shape)
print("Validation shape:", X_val_nn.shape)
print("Training labels:", y_train_nn.shape)
print("Validation labels:", y_val_nn.shape)


Preparing training vectors...
Preparing validation vectors...

Word2Vec Features Prepared!
Training shape: (127656, 100)
Validation shape: (15957, 100)
Training labels: (127656,)
Validation labels: (15957,)


In [14]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import numpy as np
import joblib
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Training device:", device)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train_nn
).astype(np.float32)

X_val_scaled = scaler.transform(
    X_val_nn
).astype(np.float32)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_scaled),
    torch.from_numpy(y_train_nn)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

class ChatNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(100, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.network(x)

model_nn = ChatNeuralNetwork().to(device)

counts = np.bincount(y_train_nn)
weights = len(y_train_nn) / (2 * counts)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.SGD(
    model_nn.parameters(),
    lr=0.02,
    momentum=0.9
)

X_val_tensor = torch.from_numpy(
    X_val_scaled
).to(device)

best_f1 = -1
start_time = time.perf_counter()

print("\nStarting Neural Network Training...\n")

for epoch in range(8):
    model_nn.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model_nn(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model_nn.eval()

    with torch.no_grad():
        val_outputs = model_nn(X_val_tensor)
        val_predictions_nn = (
            torch.argmax(val_outputs, dim=1)
            .cpu()
            .numpy()
        )

    val_f1 = f1_score(
        y_val_nn,
        val_predictions_nn,
        average="macro"
    )

    print(
        f"Epoch {epoch+1}/8 | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Validation Macro-F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            model_nn.state_dict(),
            "chat_neural_network.pt"
        )

training_time_nn = time.perf_counter() - start_time

joblib.dump(scaler, "nn_scaler.joblib")

print("\nNeural Network Training Completed!")
print("Best Validation Macro-F1:", round(best_f1, 4))
print("Training time:", round(training_time_nn, 2), "seconds")
print("Best model saved successfully!")


Training device: cuda

Starting Neural Network Training...

Epoch 1/8 | Loss: 0.3337 | Validation Macro-F1: 0.7618
Epoch 2/8 | Loss: 0.2890 | Validation Macro-F1: 0.7419
Epoch 3/8 | Loss: 0.2790 | Validation Macro-F1: 0.7681
Epoch 4/8 | Loss: 0.2719 | Validation Macro-F1: 0.7695
Epoch 5/8 | Loss: 0.2676 | Validation Macro-F1: 0.7775
Epoch 6/8 | Loss: 0.2648 | Validation Macro-F1: 0.7883
Epoch 7/8 | Loss: 0.2619 | Validation Macro-F1: 0.7822
Epoch 8/8 | Loss: 0.2579 | Validation Macro-F1: 0.7549

Neural Network Training Completed!
Best Validation Macro-F1: 0.7883
Training time: 13.09 seconds
Best model saved successfully!


In [16]:

from google.colab import drive
from pathlib import Path
import joblib
import shutil

drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/SWE015_Chat_Moderation")
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, save_dir / "tfidf.joblib")
joblib.dump(baseline_model, save_dir / "baseline_model.joblib")

w2v_model.save(str(save_dir / "chat_word2vec.model"))

shutil.copy2(
    "chat_neural_network.pt",
    save_dir / "chat_neural_network.pt"
)

joblib.dump(scaler, save_dir / "nn_scaler.joblib")

train_df.to_csv(save_dir / "train.csv", index=False)
val_df.to_csv(save_dir / "validation.csv", index=False)
test_df.to_csv(save_dir / "test.csv", index=False)

print("Project backup completed successfully!")

for file in sorted(save_dir.iterdir()):
    print(file.name)


Mounted at /content/drive
Project backup completed successfully!
baseline_model.joblib
chat_neural_network.pt
chat_word2vec.model
nn_scaler.joblib
test.csv
tfidf.joblib
train.csv
validation.csv


## 4. Fine-tuned BERT Transformer


In [17]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "google-bert/bert-base-uncased"

print("Loading BERT tokenizer...")

bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading BERT model...")

bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Clean", 1: "Offensive"},
    label2id={"Clean": 0, "Offensive": 1}
)

sample = "Hello, how are you?"

tokens = bert_tokenizer(
    sample,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("\nBERT loaded successfully!")
print("Model:", model_name)
print("Number of classes:", bert_model.config.num_labels)
print("Tokenized input shape:", tokens["input_ids"].shape)
print("GPU available:", torch.cuda.is_available())


Loading BERT tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading BERT model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERT loaded successfully!
Model: google-bert/bert-base-uncased
Number of classes: 2
Tokenized input shape: torch.Size([1, 8])
GPU available: True


In [18]:

from datasets import Dataset
from sklearn.model_selection import train_test_split

bert_train_df, _ = train_test_split(
    train_df,
    train_size=16000,
    stratify=train_df["label"],
    random_state=42
)

bert_val_df, _ = train_test_split(
    val_df,
    train_size=3000,
    stratify=val_df["label"],
    random_state=42
)

bert_train = Dataset.from_pandas(
    bert_train_df[["comment_text", "label"]].reset_index(drop=True)
)

bert_val = Dataset.from_pandas(
    bert_val_df[["comment_text", "label"]].reset_index(drop=True)
)

def tokenize_messages(batch):
    return bert_tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=128
    )

print("Tokenizing training data...")

bert_train = bert_train.map(
    tokenize_messages,
    batched=True,
    remove_columns=["comment_text"]
)

print("Tokenizing validation data...")

bert_val = bert_val.map(
    tokenize_messages,
    batched=True,
    remove_columns=["comment_text"]
)

print("\nBERT Tokenization Completed!")
print("Training samples:", len(bert_train))
print("Validation samples:", len(bert_val))
print("Training class counts:")
print(bert_train_df["label"].value_counts())
print("Example token length:", len(bert_train[0]["input_ids"]))


Tokenizing training data...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Tokenizing validation data...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]


BERT Tokenization Completed!
Training samples: 16000
Validation samples: 3000
Training class counts:
label
0    14373
1     1627
Name: count, dtype: int64
Example token length: 15


In [19]:

import torch
import numpy as np
from pathlib import Path
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import f1_score, classification_report

class_counts = np.bincount(
    bert_train_df["label"],
    minlength=2
)

class_weights = torch.tensor(
    len(bert_train_df) / (2 * class_counts),
    dtype=torch.float32
)

def weighted_loss(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits
    weights = class_weights.to(logits.device)

    loss = torch.nn.functional.cross_entropy(
        logits,
        labels,
        weight=weights,
        reduction="sum"
    )

    if num_items_in_batch is None:
        num_items_in_batch = labels.numel()

    return loss / num_items_in_batch

def compute_bert_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

bert_checkpoint_dir = str(
    save_dir / "bert_checkpoints"
)

training_args = TrainingArguments(
    output_dir=bert_checkpoint_dir,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to="none",
    seed=42
)

data_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer
)

bert_trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train,
    eval_dataset=bert_val,
    processing_class=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_bert_metrics,
    compute_loss_func=weighted_loss
)

print("Starting BERT Fine-Tuning...")

bert_trainer.train()

bert_results = bert_trainer.evaluate()

bert_save_path = str(save_dir / "bert_finetuned")

bert_trainer.save_model(bert_save_path)
bert_tokenizer.save_pretrained(bert_save_path)

print("\nBERT Fine-Tuning Completed!")
print(
    "Best Validation Macro-F1:",
    round(bert_results["eval_macro_f1"], 4)
)
print("Model saved to Google Drive!")


Starting BERT Fine-Tuning...


Epoch,Training Loss,Validation Loss,Macro F1
1,0.291272,0.403948,0.879558
2,0.158971,0.463626,0.887186


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Macro F1
0.158971,0.463626,2,0.887186


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BERT Fine-Tuning Completed!
Best Validation Macro-F1: 0.8872
Model saved to Google Drive!


In [ ]:


from transformers import AutoModelForSequenceClassification
from pathlib import Path

bert_path = Path(save_dir) / "bert_finetuned"

print("Model folder exists:", bert_path.exists())


loaded_bert, loading_info = (
    AutoModelForSequenceClassification.from_pretrained(
        str(bert_path),
        output_loading_info=True
    )
)

missing = loading_info.get("missing_keys", [])
unexpected = loading_info.get("unexpected_keys", [])

print("\nBERT Model Verification")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

print("\nFirst missing keys:")
print(sorted(missing)[:5])

print("\nFirst unexpected keys:")
print(sorted(unexpected)[:5])

if not missing and not unexpected:
    print("\nModel weights loaded correctly!")
else:
    print("\nModel loading needs checking!")


In [22]:

# Fix the verification display error

missing_list = sorted(missing)
unexpected_list = sorted(unexpected)

print("Missing keys:", len(missing_list))
print("Unexpected keys:", len(unexpected_list))

if len(missing_list) == 0 and len(unexpected_list) == 0:
    print("BERT model weights loaded correctly!")
else:
    print("Model loading needs checking!")
    print("Missing:", missing_list[:5])
    print("Unexpected:", unexpected_list[:5])


Missing keys: 0
Unexpected keys: 0
BERT model weights loaded correctly!


In [23]:

import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

loaded_bert = loaded_bert.to(device)
loaded_bert.eval()

val_loader = DataLoader(
    bert_val,
    batch_size=32,
    collate_fn=data_collator
)

all_predictions = []
all_labels = []

print("Checking saved BERT performance...")

with torch.inference_mode():
    for batch in val_loader:
        labels = batch.pop("labels")
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = loaded_bert(**batch)
        predictions = outputs.logits.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )
        all_labels.extend(
            labels.numpy()
        )

saved_bert_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

print("\nSaved BERT Validation Completed!")
print("Original Macro-F1: 0.8872")
print("Saved Model Macro-F1:", round(saved_bert_f1, 4))

if abs(saved_bert_f1 - 0.887186) < 0.001:
    print("Saved model performance verified!")
else:
    print("Performance differs - needs investigation.")


Checking saved BERT performance...

Saved BERT Validation Completed!
Original Macro-F1: 0.8872
Saved Model Macro-F1: 0.8872
Saved model performance verified!


## 5. Dynamic INT8 quantization


In [24]:

import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from pathlib import Path

print("Loading original BERT on CPU...")

original_bert_cpu = AutoModelForSequenceClassification.from_pretrained(
    str(bert_path)
).cpu().float().eval()

print("Applying INT8 quantization...")

quantized_bert = torch.ao.quantization.quantize_dynamic(
    original_bert_cpu,
    {nn.Linear},
    dtype=torch.qint8,
    inplace=False
)

quantized_bert.eval()

quantized_layers = sum(
    isinstance(module, torch.ao.nn.quantized.dynamic.Linear)
    for module in quantized_bert.modules()
)

print("Quantized Linear layers:", quantized_layers)

sample_text = "Thank you for your help"

inputs = bert_tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

with torch.inference_mode():
    outputs = quantized_bert(**inputs)
    prediction = outputs.logits.argmax(dim=1).item()

result = "Offensive" if prediction == 1 else "Clean"

print("Sample message:", sample_text)
print("Prediction:", result)

quant_save_path = Path(save_dir) / "bert_dynamic_int8_state_dict.pt"

torch.save(
    quantized_bert.state_dict(),
    quant_save_path
)

print("\nQuantization Completed!")
print("Saved file:", quant_save_path.name)
print(
    "Saved size (MB):",
    round(quant_save_path.stat().st_size / (1024 ** 2), 2)
)


Loading original BERT on CPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Applying INT8 quantization...


/tmp/ipykernel_549/179694631.py:14: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_bert = torch.ao.quantization.quantize_dynamic(
/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.

Quantized Linear layers: 74
Sample message: Thank you for your help
Prediction: Clean

Quantization Completed!
Saved file: bert_dynamic_int8_state_dict.pt
Saved size (MB): 173.09


In [26]:

import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, classification_report

quantized_bert = quantized_bert.cpu()
quantized_bert.eval()

torch.set_num_threads(4)

quant_val_loader = DataLoader(
    bert_val,
    batch_size=16,
    collate_fn=data_collator
)

quant_predictions = []
quant_labels = []

print("Evaluating Quantized BERT on CPU...")

with torch.inference_mode():
    for step, batch in enumerate(quant_val_loader):
        labels = batch.pop("labels")

        outputs = quantized_bert(**batch)
        predictions = outputs.logits.argmax(dim=1)

        quant_predictions.extend(
            predictions.cpu().numpy()
        )
        quant_labels.extend(
            labels.cpu().numpy()
        )

        if (step + 1) % 30 == 0:
            print(
                f"Completed {step + 1}"
                f"/{len(quant_val_loader)} batches"
            )

quant_f1 = f1_score(
    quant_labels,
    quant_predictions,
    average="macro"
)

print("\nQuantized BERT Evaluation Completed!")
print("Original BERT Macro-F1:", round(saved_bert_f1, 4))
print("Quantized BERT Macro-F1:", round(quant_f1, 4))
print("F1 Difference:", round(quant_f1 - saved_bert_f1, 4))

print("\nClassification Report:")
print(classification_report(
    quant_labels,
    quant_predictions,
    target_names=["Clean", "Offensive"],
    zero_division=0
))


Evaluating Quantized BERT on CPU...
Completed 30/188 batches
Completed 60/188 batches
Completed 90/188 batches
Completed 120/188 batches
Completed 150/188 batches
Completed 180/188 batches

Quantized BERT Evaluation Completed!
Original BERT Macro-F1: 0.8872
Quantized BERT Macro-F1: 0.8763
F1 Difference: -0.0109

Classification Report:
              precision    recall  f1-score   support

       Clean       0.97      0.98      0.98      2695
   Offensive       0.83      0.73      0.78       305

    accuracy                           0.96      3000
   macro avg       0.90      0.86      0.88      3000
weighted avg       0.96      0.96      0.96      3000



## 6. Same held-out test set evaluation


In [27]:

import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path

y_test = test_df["label"].to_numpy()

print("Final test samples:", len(y_test))

print("\nTesting Model 1: Classical NLP...")

X_test_classical = tfidf.transform(
    test_df["clean_text"]
)

pred_classical = baseline_model.predict(
    X_test_classical
)

f1_classical = f1_score(
    y_test, pred_classical, average="macro"
)

acc_classical = accuracy_score(
    y_test, pred_classical
)

print("Testing Model 2: Neural Network...")

model_nn.load_state_dict(
    torch.load(
        str(save_dir / "chat_neural_network.pt"),
        map_location=device,
        weights_only=True
    )
)

model_nn.to(device)
model_nn.eval()

X_test_nn = np.array([
    text_to_vector(text)
    for text in test_df["clean_text"]
], dtype=np.float32)

X_test_scaled = scaler.transform(
    X_test_nn
).astype(np.float32)

with torch.inference_mode():
    test_outputs = model_nn(
        torch.from_numpy(X_test_scaled).to(device)
    )

    pred_nn = (
        test_outputs.argmax(dim=1)
        .cpu()
        .numpy()
    )

f1_nn = f1_score(
    y_test, pred_nn, average="macro"
)

acc_nn = accuracy_score(
    y_test, pred_nn
)

print("\nFINAL TEST RESULTS")
print("----------------------------")
print("Classical NLP Macro-F1:", round(f1_classical, 4))
print("Classical NLP Accuracy:", round(acc_classical, 4))
print("Neural Network Macro-F1:", round(f1_nn, 4))
print("Neural Network Accuracy:", round(acc_nn, 4))
print("----------------------------")
print("Final testing of Models 1 and 2 completed!")


Final test samples: 15958

Testing Model 1: Classical NLP...
Testing Model 2: Neural Network...

FINAL TEST RESULTS
----------------------------
Classical NLP Macro-F1: 0.8553
Classical NLP Accuracy: 0.9386
Neural Network Macro-F1: 0.7909
Neural Network Accuracy: 0.9009
----------------------------
Final testing of Models 1 and 2 completed!


In [28]:

import torch
import numpy as np
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

bert_test = Dataset.from_pandas(
    test_df[["comment_text", "label"]].reset_index(drop=True)
)

bert_test = bert_test.map(
    tokenize_messages,
    batched=True,
    remove_columns=["comment_text"]
)

bert_test_loader = DataLoader(
    bert_test,
    batch_size=32,
    collate_fn=data_collator
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

loaded_bert = loaded_bert.to(device)
loaded_bert.eval()

bert_test_predictions = []
bert_test_labels = []

print("Testing Original BERT...")

with torch.inference_mode():
    for step, batch in enumerate(bert_test_loader):
        labels = batch.pop("labels")

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = loaded_bert(**inputs)
        predictions = outputs.logits.argmax(dim=1)

        bert_test_predictions.extend(
            predictions.cpu().numpy()
        )
        bert_test_labels.extend(
            labels.numpy()
        )

        if (step + 1) % 100 == 0:
            print(
                f"Completed {step + 1}"
                f"/{len(bert_test_loader)} batches"
            )

f1_bert_test = f1_score(
    bert_test_labels,
    bert_test_predictions,
    average="macro"
)

acc_bert_test = accuracy_score(
    bert_test_labels,
    bert_test_predictions
)

print("\nORIGINAL BERT FINAL TEST RESULTS")
print("--------------------------------")
print("Test samples:", len(bert_test_labels))
print("Macro-F1:", round(f1_bert_test, 4))
print("Accuracy:", round(acc_bert_test, 4))
print("--------------------------------")
print("Original BERT final testing completed!")


Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

Testing Original BERT...
Completed 100/499 batches
Completed 200/499 batches
Completed 300/499 batches
Completed 400/499 batches

ORIGINAL BERT FINAL TEST RESULTS
--------------------------------
Test samples: 15958
Macro-F1: 0.8972
Accuracy: 0.9622
--------------------------------
Original BERT final testing completed!


In [29]:

import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

torch.set_num_threads(4)
quantized_bert = quantized_bert.cpu()
quantized_bert.eval()

quant_test_loader = DataLoader(
    bert_test,
    batch_size=16,
    collate_fn=data_collator
)

quant_test_predictions = []
quant_test_labels = []

print("Testing Quantized BERT on CPU...")

with torch.inference_mode():
    for step, batch in enumerate(quant_test_loader):
        labels = batch.pop("labels")

        outputs = quantized_bert(**batch)
        predictions = outputs.logits.argmax(dim=1)

        quant_test_predictions.extend(
            predictions.cpu().numpy()
        )
        quant_test_labels.extend(
            labels.cpu().numpy()
        )

        if (step + 1) % 200 == 0:
            print(
                f"Completed {step + 1}"
                f"/{len(quant_test_loader)} batches"
            )

f1_quant_test = f1_score(
    quant_test_labels,
    quant_test_predictions,
    average="macro"
)

acc_quant_test = accuracy_score(
    quant_test_labels,
    quant_test_predictions
)

print("\nQUANTIZED BERT FINAL TEST RESULTS")
print("--------------------------------")
print("Test samples:", len(quant_test_labels))
print("Macro-F1:", round(f1_quant_test, 4))
print("Accuracy:", round(acc_quant_test, 4))
print(
    "F1 Difference from Original BERT:",
    round(f1_quant_test - f1_bert_test, 4)
)

print("\nClassification Report:")
print(classification_report(
    quant_test_labels,
    quant_test_predictions,
    target_names=["Clean", "Offensive"],
    zero_division=0
))

print("Quantized BERT final testing completed!")


Testing Quantized BERT on CPU...
Completed 200/998 batches
Completed 400/998 batches
Completed 600/998 batches
Completed 800/998 batches

QUANTIZED BERT FINAL TEST RESULTS
--------------------------------
Test samples: 15958
Macro-F1: 0.8983
Accuracy: 0.9643
F1 Difference from Original BERT: 0.0011

Classification Report:
              precision    recall  f1-score   support

       Clean       0.98      0.99      0.98     14335
   Offensive       0.86      0.78      0.82      1623

    accuracy                           0.96     15958
   macro avg       0.92      0.88      0.90     15958
weighted avg       0.96      0.96      0.96     15958

Quantized BERT final testing completed!


## 7. Model size and CPU latency benchmark


In [30]:

from pathlib import Path

def size_mb(paths):
    total_bytes = 0

    for path in paths:
        path = Path(path)

        if path.is_file():
            total_bytes += path.stat().st_size
        elif path.is_dir():
            total_bytes += sum(
                f.stat().st_size
                for f in path.rglob("*")
                if f.is_file()
            )

    return round(total_bytes / (1024 ** 2), 2)


classical_size = size_mb([
    save_dir / "tfidf.joblib",
    save_dir / "baseline_model.joblib"
])

nn_size = size_mb([
    save_dir / "chat_word2vec.model",
    save_dir / "chat_neural_network.pt",
    save_dir / "nn_scaler.joblib"
])


bert_size = size_mb([bert_path])


metadata_files = [
    bert_path / name
    for name in [
        "config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.txt"
    ]
    if (bert_path / name).exists()
]

quant_size = size_mb(
    [save_dir / "bert_dynamic_int8_state_dict.pt"]
    + metadata_files
)

print("MODEL SIZE RESULTS")
print("-----------------------------")
print("Classical NLP:", classical_size, "MB")
print("Neural Network:", nn_size, "MB")
print("Original BERT:", bert_size, "MB")
print("Quantized BERT:", quant_size, "MB")
print("-----------------------------")

print("Model size measurement completed!")


MODEL SIZE RESULTS
-----------------------------
Classical NLP: 1.34 MB
Neural Network: 57.14 MB
Original BERT: 418.35 MB
Quantized BERT: 173.77 MB
-----------------------------
Model size measurement completed!


In [31]:

import time
import torch
import numpy as np
import pandas as pd
import platform

torch.set_num_threads(4)

model_nn = model_nn.cpu().eval()
loaded_bert = loaded_bert.cpu().float().eval()
quantized_bert = quantized_bert.cpu().eval()

sample_message = "You are being rude and disrespectful."

def predict_classical():
    text = clean_text(sample_message)
    features = tfidf.transform([text])
    return baseline_model.predict(features)

def predict_neural():
    text = clean_text(sample_message)
    vector = text_to_vector(text).reshape(1, -1)
    features = scaler.transform(vector).astype(np.float32)

    with torch.inference_mode():
        return model_nn(
            torch.from_numpy(features)
        ).argmax(dim=1)

def predict_bert():
    inputs = bert_tokenizer(
        sample_message,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.inference_mode():
        return loaded_bert(
            **inputs
        ).logits.argmax(dim=1)

def predict_quantized():
    inputs = bert_tokenizer(
        sample_message,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.inference_mode():
        return quantized_bert(
            **inputs
        ).logits.argmax(dim=1)

def benchmark(name, function, runs=100):
    print(f"\nTesting {name}...", flush=True)

    for _ in range(3):
        function()

    times = []

    for i in range(runs):
        start = time.perf_counter()
        function()
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)

        if (i + 1) % 25 == 0:
            print(f"Completed {i+1}/{runs}", flush=True)

    average = np.mean(times)
    print(f"Average latency: {average:.2f} ms", flush=True)

    return round(float(average), 2)

with open("/proc/cpuinfo") as f:
    cpu_lines = f.readlines()

cpu_name = next(
    (line.split(":", 1)[1].strip()
     for line in cpu_lines
     if line.startswith("model name")),
    platform.processor() or "Unknown CPU"
)

print("CPU:", cpu_name)
print("PyTorch:", torch.__version__)
print("Runs per model: 100")

latency_classical = benchmark(
    "Classical NLP", predict_classical
)

latency_nn = benchmark(
    "Neural Network", predict_neural
)

latency_bert = benchmark(
    "Original BERT", predict_bert
)

latency_quant = benchmark(
    "Quantized BERT", predict_quantized
)

results_table = pd.DataFrame({
    "Model": [
        "Classical NLP",
        "Neural Network",
        "Original BERT",
        "Quantized BERT"
    ],
    "Macro-F1": [
        f1_classical,
        f1_nn,
        f1_bert_test,
        f1_quant_test
    ],
    "Size (MB)": [
        classical_size,
        nn_size,
        bert_size,
        quant_size
    ],
    "Latency (ms)": [
        latency_classical,
        latency_nn,
        latency_bert,
        latency_quant
    ]
})

results_table.to_csv(
    save_dir / "final_results.csv",
    index=False
)

print("\nFINAL MODEL COMPARISON")
print(results_table.to_string(index=False))
print("\nResults saved to Google Drive!")
print("Benchmark CPU:", cpu_name)


CPU: Intel(R) Xeon(R) CPU @ 2.00GHz
PyTorch: 2.14.1+cu130
Runs per model: 100

Testing Classical NLP...
Completed 25/100
Completed 50/100
Completed 75/100
Completed 100/100
Average latency: 2.28 ms

Testing Neural Network...
Completed 25/100
Completed 50/100
Completed 75/100
Completed 100/100
Average latency: 1.23 ms

Testing Original BERT...
Completed 25/100
Completed 50/100
Completed 75/100
Completed 100/100
Average latency: 210.72 ms

Testing Quantized BERT...
Completed 25/100
Completed 50/100
Completed 75/100
Completed 100/100
Average latency: 61.55 ms

FINAL MODEL COMPARISON
         Model  Macro-F1  Size (MB)  Latency (ms)
 Classical NLP  0.855268       1.34          2.28
Neural Network  0.790921      57.14          1.23
 Original BERT  0.897214     418.35        210.72
Quantized BERT  0.898324     173.77         61.55

Results saved to Google Drive!
Benchmark CPU: Intel(R) Xeon(R) CPU @ 2.00GHz


## 8. Interactive Gradio demo in Colab


In [37]:

import gradio as gr
import torch
import numpy as np
import time

# Prepare all four models for CPU inference
torch.set_num_threads(4)
model_nn.cpu().eval()
loaded_bert.cpu().float().eval()
quantized_bert.cpu().eval()

model_choices = [
    "Quantized BERT (INT8)",
    "Original BERT (FP32)",
    "Neural Network (Word2Vec)",
    "Classical NLP (TF-IDF)"
]

# Predict using the selected trained model
def predict_message(message, selected_model):
    start = time.perf_counter()

    if selected_model == "Classical NLP (TF-IDF)":
        cleaned = clean_text(message)
        features = tfidf.transform([cleaned])
        probabilities = baseline_model.predict_proba(features)[0]

    elif selected_model == "Neural Network (Word2Vec)":
        cleaned = clean_text(message)
        vector = text_to_vector(cleaned).reshape(1, -1)
        scaled = scaler.transform(vector).astype(np.float32)

        with torch.inference_mode():
            logits = model_nn(torch.from_numpy(scaled))
            probabilities = torch.softmax(
                logits, dim=1
            )[0].numpy()

    else:
        inputs = bert_tokenizer(
            message,
            return_tensors="pt",
            truncation=True,
            max_length=128
        )

        model = (
            quantized_bert
            if selected_model == "Quantized BERT (INT8)"
            else loaded_bert
        )

        with torch.inference_mode():
            logits = model(**inputs).logits
            probabilities = torch.softmax(
                logits, dim=1
            )[0].numpy()

    latency = (time.perf_counter() - start) * 1000

    prediction = int(np.argmax(probabilities))
    confidence = float(probabilities[prediction]) * 100

    return prediction, confidence, latency


# Mask messages classified as offensive
def mask_message(message):
    return "".join(
        "*" if character.isalnum() else character
        for character in message
    )


# Display the response-time log
def format_logs(logs):
    if not logs:
        return "No messages tested yet."

    return "\n".join(
        f"{i+1}. {entry['model']}\n"
        f"   {entry['label']} | {entry['latency']:.2f} ms"
        for i, entry in enumerate(logs)
    )


# Process each new chat message
def moderate_chat(message, selected_model, history, logs):
    history = list(history or [])
    logs = list(logs or [])

    if not message.strip():
        return message, history, logs, format_logs(logs), \
            "Please enter a message."

    prediction, confidence, latency = predict_message(
        message, selected_model
    )

    label = "Offensive" if prediction == 1 else "Clean"
    displayed = mask_message(message) if prediction == 1 else message

    # Rule-based detection is shown separately
    detected_words = check_offensive_words(message)

    history.append({
        "role": "user",
        "content": displayed
    })

    response = (
        f"**Classification:** {label}\n\n"
        f"**Model confidence:** {confidence:.2f}%\n\n"
        f"**Response time:** {latency:.2f} ms\n\n"
        f"**Model:** {selected_model}"
    )



    if detected_words:
        response += (
            f"\n\n**Rule-based matches detected:** "
            f"{len(detected_words)}"
        )

    history.append({
        "role": "assistant",
        "content": response
    })

    logs.append({
        "model": selected_model,
        "label": label,
        "latency": latency
    })

    return (
        "",
        history,
        logs,
        format_logs(logs),
        f"Last result: {label} | {confidence:.2f}% confidence"
    )



# Reset the interface
def reset_chat():
    return "", [], [], "No messages tested yet.", "No classification yet."


# Build the real interactive application
with gr.Blocks(title="SWE015 - AI Chat Moderation") as demo:

    gr.Markdown("# AI Chat Moderation System")
    gr.Markdown(
        "Real-time message classification using Classical NLP, "
        "Neural Networks, BERT, and Quantized BERT."
    )

    with gr.Row():

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Live Moderated Chat",
                height=430
            )

            message_input = gr.Textbox(
                label="Your Message",
                placeholder="Type a message to check...",
                lines=2
            )

            with gr.Row():
                send_button = gr.Button(
                    "Check Message",
                    variant="primary"
                )
                clear_button = gr.Button("Clear Chat")

        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=model_choices,
                value="Quantized BERT (INT8)",
                label="Select Model"
            )

            last_result = gr.Markdown(
                "No classification yet."
            )

            gr.Markdown("### Response Time Log")

            timing_log = gr.Textbox(
                value="No messages tested yet.",
                lines=12,
                interactive=False,
                label="Message Processing History"
            )

    chat_logs = gr.State([])

    inputs = [
        message_input,
        model_selector,
        chatbot,
        chat_logs
    ]

    outputs = [
        message_input,
        chatbot,
        chat_logs,
        timing_log,
        last_result
    ]

    send_button.click(
        fn=moderate_chat,
        inputs=inputs,
        outputs=outputs
    )

    message_input.submit(
        fn=moderate_chat,
        inputs=inputs,
        outputs=outputs
    )

    clear_button.click(
        fn=reset_chat,
        inputs=[],
        outputs=outputs
    )

# Launch the application
demo.launch(share=True, debug=True)



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e57caa6d60a43124b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e57caa6d60a43124b5.gradio.live


## 9. Quantized model reload verification


In [38]:

import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification

print("Loading saved Quantized BERT...")

reloaded_original = AutoModelForSequenceClassification.from_pretrained(
    str(bert_path)
).cpu().float().eval()

reloaded_quantized = torch.ao.quantization.quantize_dynamic(
    reloaded_original,
    {nn.Linear},
    dtype=torch.qint8
)

saved_weights = torch.load(
    str(save_dir / "bert_dynamic_int8_state_dict.pt"),
    map_location="cpu",
    weights_only=True
)

reloaded_quantized.load_state_dict(
    saved_weights,
    strict=True
)

reloaded_quantized.eval()

test_input = bert_tokenizer(
    "Thank you for your help",
    return_tensors="pt",
    truncation=True,
    max_length=128
)

with torch.inference_mode():
    old_output = quantized_bert(**test_input).logits
    new_output = reloaded_quantized(**test_input).logits

same_results = torch.allclose(
    old_output,
    new_output,
    atol=1e-5,
    rtol=1e-5
)

print("\nQuantized model loaded successfully!")
print("Predictions match:", same_results)

if same_results:
    print("Saved Quantized BERT is ready for packaging!")
else:
    print("Model outputs differ - needs checking.")


Loading saved Quantized BERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_549/2451823656.py:11: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  reloaded_quantized = torch.ao.quantization.quantize_dynamic(
/usr/local/lib/python3.13/dist-packages/torch/_utils.py:466: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are u


Quantized model loaded successfully!
Predictions match: True
Saved Quantized BERT is ready for packaging!


## 10. Saved files and export


In [39]:

from pathlib import Path

github_dir = save_dir / "github_project"
github_dir.mkdir(parents=True, exist_ok=True)

required_files = {
    "Classical NLP": [
        save_dir / "tfidf.joblib",
        save_dir / "baseline_model.joblib"
    ],
    "Neural Network": [
        save_dir / "chat_word2vec.model",
        save_dir / "chat_neural_network.pt",
        save_dir / "nn_scaler.joblib"
    ],
    "Original BERT": [
        save_dir / "bert_finetuned"
    ],
    "Quantized BERT": [
        save_dir / "bert_dynamic_int8_state_dict.pt"
    ],
    "Final Results": [
        save_dir / "final_results.csv"
    ]
}

print("Checking project files...\n")

all_ready = True

for model_name, files in required_files.items():
    available = all(path.exists() for path in files)

    print(
        model_name + ":",
        "READY" if available else "MISSING"
    )

    if not available:
        all_ready = False

print("\nGitHub project folder:", github_dir)

if all_ready:
    print("\nAll project files are ready!")
else:
    print("\nSome files are missing.")


Checking project files...

Classical NLP: READY
Neural Network: READY
Original BERT: READY
Quantized BERT: READY
Final Results: READY

GitHub project folder: /content/drive/MyDrive/SWE015_Chat_Moderation/github_project

All project files are ready!


In [1]:

from google.colab import drive
from pathlib import Path
import zipfile

# Connect Google Drive
drive.mount("/content/drive")

# Project folder
project = Path("/content/drive/MyDrive/SWE015_Chat_Moderation")

# Files needed for the application
files = [
    "tfidf.joblib",
    "baseline_model.joblib",
    "chat_word2vec.model",
    "chat_neural_network.pt",
    "nn_scaler.joblib",
    "bert_dynamic_int8_state_dict.pt"
]

bert_folder = project / "bert_finetuned"

# Check all files before creating the ZIP
missing = [name for name in files if not (project / name).is_file()]

if not bert_folder.is_dir():
    missing.append("bert_finetuned")

if missing:
    print("Missing files:", missing)
else:
    zip_path = project / "SWE015_models.zip"

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True
    ) as archive:

        for name in files:
            archive.write(
                project / name,
                arcname=f"models/{name}"
            )

        for file in bert_folder.rglob("*"):
            if file.is_file():
                archive.write(
                    file,
                    arcname=str(
                        Path("models") / "bert_finetuned"
                        / file.relative_to(bert_folder)
                    )
                )

    print("ZIP created successfully!")
    print("File:", zip_path.name)
    print("Size (MB):", round(zip_path.stat().st_size / 1024**2, 2))
    print("Saved in Google Drive!")


Mounted at /content/drive
ZIP created successfully!
File: SWE015_models.zip
Size (MB): 649.94
Saved in Google Drive!
